# Stage 2: Multi-Agent Manipulation Detection System

Notebook này sử dụng hệ thống multi-agent với LangGraph để phát hiện manipulation trong các cuộc đối thoại pháp lý.

## Kiến trúc hệ thống:
1. **Detector Agent**: Phát hiện có manipulation hay không
2. **Analyzer Agent**: Xác định manipulator và kỹ thuật
3. **Evidence Agent**: Xác thực bằng chứng
4. **Meta Agent**: Tổng hợp kết quả cuối cùng

## Model:
- Base: Mistral-7B-Instruct-v0.3 (4-bit quantized)
- Fine-tuned: Custom adapter từ stage 1

## 1. Cài đặt thư viện

Cài đặt các thư viện cần thiết cho multi-agent system và model inference.

In [1]:
!pip install protobuf==3.20.3 --quiet
!pip install -U langgraph transformers bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

## 2. Import thư viện

In [2]:
import json 
import pandas as pd 
import torch 
from typing import Dict, Any, Optional, List, Union, TypedDict 
import re
from langchain_core.messages import HumanMessage, AIMessage 
from langchain_core.tools import tool 
from langgraph.checkpoint.memory import MemorySaver 
from langgraph.graph import END, START, StateGraph 
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

2026-01-20 13:03:02.525562: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768914182.765840      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768914182.831796      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## 3. Cấu hình HuggingFace Token

Lấy HF token từ Kaggle secrets để tải model từ HuggingFace Hub.

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

## 4. Kiểm tra môi trường

Kiểm tra CUDA, GPU memory và PyTorch version trước khi load model.

In [4]:
# Kiểm tra môi trường CUDA
print("=== Environment Check ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch Version: {torch.__version__}")
print("=" * 50)

=== Environment Check ===
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 15.83 GB
PyTorch Version: 2.6.0+cu124


## 5. Load Base Model

Tải Mistral-7B-Instruct với cấu hình:
- **4-bit quantization** (NF4): Giảm memory footprint
- **BitsAndBytes**: Quantization backend
- **Device map auto**: Tự động phân bổ model lên GPU/CPU

In [5]:
torch.cuda.empty_cache() 

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading tokenizer from {base_model_id}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded")

print(f"Loading base model from {base_model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)
print("Base model loaded in 4-bit")

Loading tokenizer from mistralai/Mistral-7B-Instruct-v0.3...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Base model loaded in 4-bit


## 6. Load Fine-tuned Adapter & Merge

Load PEFT adapter đã được fine-tune từ stage 1:
- Kiểm tra path tồn tại trước khi load
- **Merge model** để tạo standalone model
- Set model về `eval()` mode cho inference

⚠️ **Lưu ý**: Merge sẽ cần nhiều GPU memory hơn. Nếu bị OOM, bỏ dòng merge.

In [6]:
import os

adapter_model_id = "/kaggle/input/final-model2/final_model"  # add finetuned model

# Kiểm tra path tồn tại
if not os.path.exists(adapter_model_id):
    raise FileNotFoundError(f"Adapter path not found: {adapter_model_id}")

print(f"Found adapter at: {adapter_model_id}")

try:
    model = PeftModel.from_pretrained(base_model, adapter_model_id)
    print("PEFT adapter loaded successfully")
    
except Exception as e:
    print(f"Failed to load/merge PEFT adapter: {e}")
    raise

Found adapter at: /kaggle/input/final-model2/final_model
PEFT adapter loaded successfully


## 7. Load Test Data

Load dataset chứa:
- **Dialogue**: Cuộc hội thoại giữa plaintiff và defendant

In [7]:
def load_csv(file_path):
    return pd.read_csv(file_path)

csv_df = load_csv("/kaggle/input/intent/test_intent.csv")
dialogues = csv_df["Dialogue"].tolist()

## 8. Định nghĩa Manipulation Tactics

Danh sách 11 kỹ thuật manipulation được model nhận diện

In [8]:
allowed_tactics = [
    "persuasion", "playing the victim", "gaslighting", "evasion", 
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

## 9. Hàm Generation

Hàm sinh text từ model với:
- **torch.no_grad()**: Tắt gradient để tiết kiệm bộ nhớ
- **model.eval()**: Chế độ inference
- **Proper tokenization**: Pad token, EOS token
- **Temperature 0.4**: Cân bằng giữa creativity và consistency

In [9]:
def generate_response(prompt, max_new_tokens=1000, temperature=0.4):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=False
    )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

## 10. Agent State Definition

TypedDict định nghĩa state được chia sẻ giữa các agents:
- Input: dialogue, intents
- Intermediate: manipulation_detected, primary_manipulator, techniques
- Output: final_summary
- Messages: Lưu lịch sử conversation

In [10]:
class AgentState(TypedDict):
    dialogue: str
    dialogue_index: int
    manipulation_detected: Optional[str]
    primary_manipulator: Optional[str]
    techniques_used: Optional[List[str]]
    evidence: Optional[str]
    final_summary: Optional[Dict[str, Any]]
    messages: List[Union[HumanMessage, AIMessage]]

## 11. Detector Agent

**Agent đầu tiên** trong pipeline:
- **Nhiệm vụ**: Phát hiện có manipulation hay không
- **Output**: Yes/No
- **Parsing**: Regex robust để tìm Yes/No trong response
- **Routing**: Yes → Analyzer Agent, No → Meta Agent (skip analysis)

In [11]:
def detector_agent(state):
    dialogue = state["dialogue"]
    system_prompt = f"""<s>[INST] You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Analyze it thoroughly.
3. Determine if any manipulation is present or not.

Dialogue: {dialogue}

Question: Is there manipulation present in this dialogue?
You must respond with ONLY one word: Yes or No
[/INST]"""

    response = generate_response(system_prompt, max_new_tokens=5)  
    print(f"Raw LLM response: {response}")

    model_response = re.search(r"\[/INST\](.*?)$", response, re.DOTALL)
    if model_response:
        extracted_response = model_response.group(1).strip()
    else:
        extracted_response = response.strip()
    
    print(f"Full extracted response: {extracted_response}")
    lines = [line.strip() for line in extracted_response.split('\n') if line.strip()]
    if lines:
        last_line = lines[-1]
        print(f"Last non-empty line: {last_line}")
        
        if last_line.lower() == "yes":
            result = "Yes"
            print("Last line is exactly 'Yes'")
        elif last_line.lower() == "no":
            result = "No"
            print("Last line is exactly 'No'")
        else:
            result = "No"
            print("Last line is not exactly 'Yes' or 'No', defaulting to No")
    else:
        result = "No"
        print("No non-empty lines found, defaulting to No")
    
    print(f"Final result after parsing: {result}")
    
    return {
        **state,
        "manipulation_detected": result,
        "messages": state["messages"] + [AIMessage(content=f"Manipulation Present: {result}")]
    }

## 12. Analyzer Agent

**Agent thứ hai** (chỉ chạy nếu có manipulation):
- **Nhiệm vụ**: 
  - Xác định primary manipulator (Plaintiff/Defendant)
  - Nhận diện manipulation techniques được sử dụng
- **Input**: Dialogue + intents của cả hai bên
- **Output**: Primary manipulator + list of techniques

In [12]:
def analyzer_agent(state):   
    if state["manipulation_detected"] == "No":
        return {
            **state,
            "primary_manipulator": "None",
            "techniques_used": [ ],
            "messages": state["messages"] + [AIMessage(content="No manipulation detected, skipping analysis.")]
        }
    
    print("Manipulation detected, analyzing manipulator and techniques...")
    combined_prompt = f"""<s>[INST] Compare both dialogue, and analyze the manipulation:
    
Dialogue: {state['dialogue']}

First, determine who is the primary manipulator (answer with ONLY Plaintiff or Defendant).
Second, identify which manipulation techniques they used from this list:
{', '.join(allowed_tactics)}

Format your response exactly like this:
Primary Manipulator: [ ]
Techniques: [list ONLY the techniques that were actually used, separated by commas]
[/INST]"""

    print("Prompting model for manipulation analysis...")
    combined_response = generate_response(combined_prompt, max_new_tokens=100)
    print(f"Raw analyzer response: {combined_response}")
    primary_manipulator = "Unknown"
    if "primary manipulator: plaintiff" in combined_response.lower():
        primary_manipulator = "Plaintiff"
    elif "primary manipulator: defendant" in combined_response.lower():
        primary_manipulator = "Defendant"
    techniques_used = []
    for line in combined_response.split('\n'):
        if line.lower().startswith("techniques:"):
            techniques_part = line.split(':', 1)[1].strip()
            for technique in techniques_part.split(','):
                technique = technique.strip()
                for allowed_tactic in allowed_tactics:
                    if technique.lower() == allowed_tactic.lower():
                        techniques_used.append(allowed_tactic)
                        break
    
    analysis_result = f"Primary Manipulator: {primary_manipulator}\nManipulative Techniques: {', '.join(techniques_used) if techniques_used else 'None'}"
    
    print(f"ANALYZER AGENT RESULT:\n{analysis_result}")
    
    return {
        **state,
        "primary_manipulator": primary_manipulator,
        "techniques_used": techniques_used,
        "messages": state["messages"] + [AIMessage(content=analysis_result)]
    }

## 13. Evidence Agent

**Agent thứ ba** - Validation layer:
- **Nhiệm vụ**: Xác thực kết quả từ Analyzer Agent với bằng chứng cụ thể
- **Cross-check**: So sánh manipulation techniques với dialogue thực tế
- **Refinement**: Loại bỏ techniques không có bằng chứng rõ ràng
- **Output**: Confirmed manipulator + confirmed techniques

In [13]:
def evidence_agent(state):
    dialogue = state["dialogue"]
    primary_manipulator = state["primary_manipulator"]
    techniques_used = state["techniques_used"]
    prompt = f"""<s>[INST] Analyze this dialogue for manipulation.

Dialogue: {dialogue}
Current analysis:
- Primary Manipulator: {primary_manipulator}
- Techniques identified: {', '.join(techniques_used) if techniques_used else 'None'}

For the current analysis, validate if the primary manipulator is correct and which techniques are actually used with clear evidence.

Format your response EXACTLY like this:
Confirmed Manipulator: [ ]
Confirmed Techniques: [ ]

Only include techniques from this list with clear evidence: {', '.join(allowed_tactics)}
[/INST]"""

    print("Prompting model for evidence validation...")
    validation_response = generate_response(prompt, max_new_tokens=500)
    print(f"Raw evidence response: {validation_response}")
    
    confirmed_manipulator = primary_manipulator  
    confirmed_techniques = []
    
    if "confirmed manipulator: plaintiff" in validation_response.lower():
        confirmed_manipulator = "Plaintiff"
    elif "confirmed manipulator: defendant" in validation_response.lower():
        confirmed_manipulator = "Defendant"
    
    for line in validation_response.split('\n'):
        if line.lower().startswith("confirmed techniques:"):
            techniques_part = line.split(':', 1)[1].strip()
            techniques_part = techniques_part.strip('[]')
            
            if techniques_part.lower() != "none":
                for technique in techniques_part.split(','):
                    technique = technique.strip()
                    for allowed_tactic in allowed_tactics:
                        if technique.lower() == allowed_tactic.lower():
                            confirmed_techniques.append(allowed_tactic)
                            break
    if not confirmed_techniques:
        for tactic in allowed_tactics:
            pattern = r'[0-9.•\-\*]+\s*' + re.escape(tactic) + r'[:\s]'
            if re.search(pattern, validation_response, re.IGNORECASE):
                confirmed_techniques.append(tactic)
    techniques_string = ', '.join(confirmed_techniques) if confirmed_techniques else 'None'
    
    validation_result = f"""Evidence Assessment:
Confirmed Manipulator: {confirmed_manipulator}
Confirmed Techniques: {techniques_string}
"""
    
    print(f"EVIDENCE AGENT RESULT:\nManipulator: {confirmed_manipulator}\nTechniques: {confirmed_techniques}")
    
    return {
        **state,
        "primary_manipulator": confirmed_manipulator,
        "techniques_used": confirmed_techniques,
        "evidence": validation_result,
        "messages": state["messages"] + [AIMessage(content=validation_result)]
    }

## 14. Meta Agent

**Agent cuối cùng** - Aggregation & Decision:
- **Nhiệm vụ**: Tổng hợp kết quả từ tất cả agents
- **Logic**: 
  - Nếu có manipulation + manipulator + techniques → Yes
  - Ngược lại → No
- **Output**: Final summary với 3 fields:
  - manipulation_present: Yes/No
  - primary_manipulator: Plaintiff/Defendant/None
  - techniques: List of confirmed techniques

In [14]:
def meta_agent(state):
    
    manipulation_present = state["manipulation_detected"]
    primary_manipulator = state["primary_manipulator"]
    techniques = state["techniques_used"]
    
    print(f"Preparing final summary from:\nManipulation: {manipulation_present}\nManipulator: {primary_manipulator}\nTechniques: {techniques}")
    if manipulation_present == "Yes" and primary_manipulator not in ["None", "Unknown"] and techniques:
        final_manipulation = "Yes"
        final_manipulator = primary_manipulator
        final_techniques = techniques
    else:
        if manipulation_present == "Yes" and primary_manipulator not in ["None", "Unknown"] and not techniques:
            analyzer_detected_techniques = state.get("techniques_used", [])
            if analyzer_detected_techniques:
                final_manipulation = "Yes"
                final_manipulator = primary_manipulator
                final_techniques = analyzer_detected_techniques
            else:
                final_manipulation = "No"
                final_manipulator = "None"
                final_techniques = []
        else:
            final_manipulation = "No"
            final_manipulator = "None"
            final_techniques = []
    
    formatted_response = f"""Manipulation Present: {final_manipulation}
Primary Manipulator: {final_manipulator}
Manipulative Techniques: {", ".join(final_techniques) if final_techniques else "None"}"""

    final_result = {
        "manipulation_present": final_manipulation,
        "primary_manipulator": final_manipulator,
        "techniques": final_techniques
    }
    
    print(f"META AGENT FINAL OUTPUT:\n{formatted_response}")
    
    return {
        **state,
        "final_summary": final_result,
        "messages": state["messages"] + [AIMessage(content=formatted_response)]
    }

## 15. Workflow Graph & Processing Functions

### LangGraph Workflow:
- **StateGraph**: Định nghĩa flow giữa các agents
- **Conditional edges**: Routing logic (Yes/No branching)
- **Entry point**: Detector Agent
- **End point**: Meta Agent

### Processing Functions:
- `process_dialogue()`: Xử lý 1 dialogue qua toàn bộ pipeline
- `save_results_to_csv()`: Xử lý tất cả dialogues và lưu kết quả

In [15]:
def router(state):
    if "final_summary" in state:
        return END
    
    if "evidence" in state:
        return "meta_agent"
    
    if "primary_manipulator" in state:
        return "evidence_agent"
        
    if "manipulation_detected" in state:
        return "analyzer_agent"
    
    return "detector_agent"

workflow_graph = StateGraph(AgentState)

workflow_graph.add_node("detector_agent", detector_agent)
workflow_graph.add_node("analyzer_agent", analyzer_agent)
workflow_graph.add_node("evidence_agent", evidence_agent)
workflow_graph.add_node("meta_agent", meta_agent)

workflow_graph.add_conditional_edges(
    "detector_agent",
    lambda state: "analyzer_agent" if state["manipulation_detected"] == "Yes" else "meta_agent"
)

workflow_graph.add_edge("analyzer_agent", "evidence_agent")
workflow_graph.add_edge("evidence_agent", "meta_agent")
workflow_graph.add_edge("meta_agent", END)
workflow_graph.set_entry_point("detector_agent")
workflow = workflow_graph.compile()


def process_dialogue(dialogue_idx):
    dialogue = dialogues[dialogue_idx]

    initial_state = AgentState({
        "dialogue": dialogue,
        "dialogue_index": dialogue_idx,
        "manipulation_detected": None,
        "primary_manipulator": None,
        "techniques_used": None,
        "evidence": None,
        "final_summary": None,
        "messages": [HumanMessage(content=f"Analyze this dialogue for manipulation: {dialogue}")]
    })
    final_state = workflow.invoke(initial_state)
    return final_state


def save_results_to_csv():
    manipulation_results = []
    manipulator_results = []
    techniques_results = []

    for idx in range(len(dialogues)):
        print(f"\n=== Processing Dialogue {idx} ===")
        result = process_dialogue(idx)
        manipulation_results.append(result['final_summary']['manipulation_present'])
        manipulator_results.append(result['final_summary']['primary_manipulator'])
        techniques_results.append(', '.join(result['final_summary']['techniques']) 
                                if result['final_summary']['techniques'] else 'None')
        print(f"Final Summary:")
        print(f"Manipulation Present: {result['final_summary']['manipulation_present']}")
        print(f"Primary Manipulator: {result['final_summary']['primary_manipulator']}")
        print(f"Techniques Used: {', '.join(result['final_summary']['techniques']) if result['final_summary']['techniques'] else 'None'}")

    csv_df['Manipulation_Present'] = manipulation_results
    csv_df['Primary_Manipulator'] = manipulator_results
    csv_df['Techniques_Used'] = techniques_results
    output_path = "/kaggle/working/manipulation_results.csv"
    csv_df.to_csv(output_path, index=False)
    

    print(f"Results saved to: {output_path}")
    return output_path

## 16. Run Full Pipeline

In [16]:
if __name__ == "__main__":
    save_results_to_csv()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



=== Processing Dialogue 0 ===


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw LLM response: You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Analyze it thoroughly.
3. Determine if any manipulation is present or not.

Dialogue: Judge
I understand, but my hypothetical assumes that you know all that, your clients know all that, maybe even intend it.

Now whether there -- there are allegations in this complaint sufficient, put that aside.

But, if you know or intend all of that, then what?

Lawyer of Defendant
I still don't think that establishes proximate cause when you have an intervening independent crime.

And I'll point you to Hemi Group, Your Honor.

Judge
Yeah.

Lawyer of Defendant
In Hemi Group, the underlying statute was the filing of the tax reports.

The entire purpose of the tax reports was to allow governmental entities to collect tax revenue from cigarette sales online that weren't otherwise subject to sales taxes. But

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Raw LLM response: You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Analyze it thoroughly.
3. Determine if any manipulation is present or not.

Dialogue: Defendant's Lawyer: We don't disagree that marriage is an important right and that she has liberty interests that are implicated there.
To step back to the beginning of your question, we think that the doctrine of consular nonreviewability is rooted in the political branch's plenary power to determine which noncitizens should be admitted to the country and what procedures are going to be used in order to make that determination.
And so that's not just a judicial exception to general principles of reviewability.
That's the background against which Congress enacted the INA. It provided for certain remedies in review and did not provide for this.

Judge: But that's the second question.

Defendant's Lawyer: An

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw LLM response: You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Analyze it thoroughly.
3. Determine if any manipulation is present or not.

Dialogue: Plaintiff: I don't think he is budgeting right. I think he's spending all his money on his new fiancée, and she's—

Judge: Oh, you're mad about that?

Plaintiff: Yeah, I'm pretty mad about it.

Judge: Okay, well, um, get over it. That’s the first thing. Second thing is, you said he has another child. So even without a reduction in what he makes, that requires an adjustment right there. Him having another child entitles him to get his child support adjusted.

Plaintiff: Well, maybe he should get his attitude adjusted. I don’t—

Question: Is there manipulation present in this dialogue?
You must respond with ONLY one word: Yes or No
 Yes
Full extracted response: You are a manipulation detection expert. Your t

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw analyzer response: Compare both dialogue, and analyze the manipulation:
    
Dialogue: Plaintiff: I don't think he is budgeting right. I think he's spending all his money on his new fiancée, and she's—

Judge: Oh, you're mad about that?

Plaintiff: Yeah, I'm pretty mad about it.

Judge: Okay, well, um, get over it. That’s the first thing. Second thing is, you said he has another child. So even without a reduction in what he makes, that requires an adjustment right there. Him having another child entitles him to get his child support adjusted.

Plaintiff: Well, maybe he should get his attitude adjusted. I don’t—

First, determine who is the primary manipulator (answer with ONLY Plaintiff or Defendant).
Second, identify which manipulation techniques they used from this list:
persuasion, playing the victim, gaslighting, evasion, deflection, minimization, dismissal, guilt tripping, emotional appeal, framing the narrative, character attack

Format your response exactly like this:
Primar